In [ ]:
# install HuggingFace 'datasets'
!pip install datasets

In [ ]:
# load wikitext from datasets
# using wikitext2 raw version

from datasets import load_dataset

dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})


In [ ]:
# total rows and non-empty rows dont match here
# need to do some cleaning
count_nonempty = 0
count_total = 0

for entry in dataset["train"]:
    count_total += 1
    if entry["text"].strip() != "":
        count_nonempty += 1

print("Total rows:", count_total)
print("Non-empty rows:", count_nonempty)

Total rows: 36718
Non-empty rows: 23767


In [ ]:
# remove blank lines
dataset = dataset.filter(lambda x: x["text"].strip() != "")

Filter:   0%|          | 0/4358 [00:00<?, ? examples/s]

Filter:   0%|          | 0/36718 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [ ]:
# checking what data looks like
print(dataset["train"][0])

{'text': ' = Valkyria Chronicles III = \n'}


In [ ]:
# now the row numbers match
count_nonempty = 0
count_total = 0

for entry in dataset["train"]:
    count_total += 1
    if entry["text"].strip() != "":
        count_nonempty += 1

print("Total rows:", count_total)
print("Non-empty rows:", count_nonempty)

Total rows: 23767
Non-empty rows: 23767


In [ ]:
# PRE-PROCESSING
# 're' for regex
# this function extracts and cleans text
# note to self: wikitext2 is about 2M words
# this counts about 1.75M
import re

def preprocess(text):
    text = text.lower()
    return re.findall(r'\b\w+\b', text)

# building the corpus
all_words = []

for entry in dataset["train"]:
    text = entry["text"]
    if text.strip():   # simple safe guard
        all_words.extend(preprocess(text))

print("Total words:", len(all_words))

Total words: 1750956


In [ ]:
# build word frequency
#count unique words and most frequent words
from collections import Counter

word_freq = Counter(all_words)

print("Unique words:", len(word_freq))
print(word_freq.most_common(10))

Unique words: 65867
[('the', 130771), ('of', 57032), ('and', 50738), ('in', 45019), ('to', 39522), ('a', 36567), ('was', 21008), ('on', 15141), ('as', 15058), ('s', 14982)]


In [ ]:
# function to do prefix based matching for suggestions
# return top 5 suggestions
def get_suggestions(prefix, word_freq, k=5):
    prefix = prefix.lower()

    matches = [word for word in word_freq if word.startswith(prefix)]

    ranked = sorted(matches, key=lambda w: word_freq[w], reverse=True)

    return ranked[:k]

In [ ]:
# sanity check
# testing suggestions for "comp" and "auto"
print(get_suggestions("comp", word_freq))
print(get_suggestions("auto", word_freq))

['company', 'completed', 'complete', 'companies', 'competition']
['autobiography', 'automobile', 'automatic', 'auto', 'autonomy']


In [ ]:
# sanity check
# testing suggestions for "a" and "b"
print(get_suggestions("a", word_freq))
print(get_suggestions("b", word_freq))

['and', 'a', 'as', 'at', 'an']
['by', 'be', 'but', 'been', 'between']


In [ ]:
def autocomplete(prefix, word_freq, k=5):
    prefix = prefix.lower()

    # Find matching words
    matches = [word for word in word_freq if word.startswith(prefix)]

    # Rank by frequency
    matches = sorted(matches, key=lambda w: word_freq[w], reverse=True)

    return matches[:k]

In [ ]:
# sanity check
# test the autocomplete function
print(autocomplete("comp", word_freq))
print(autocomplete("auto", word_freq))
print(autocomplete("data", word_freq))

['company', 'completed', 'complete', 'companies', 'competition']
['autobiography', 'automobile', 'automatic', 'auto', 'autonomy']
['data', 'database', 'datadyne', 'databases', 'dataflow']


In [ ]:
# make a random sample of 1000 words from the dataset
# print the first 10 just to check
import random

sample = random.sample(list(word_freq.keys()), 1000)
print(sample[:10])

['127th', '128749', 'semifinals', 'hemispherical', 'raccoons', 'dews', 'tiltan', 'reardon', 'brew', 'demyelinating']


In [ ]:
# EXPERIMENT STARTS HERE
def keystroke_savings_real(word, word_counts, k=5):
    for i in range(1, len(word)):
        prefix = word[:i]
        suggestions = autocomplete(prefix, word_counts, k)

        if word in suggestions:
            # user would stop typing here
            return len(word) - i

    # never predicted
    return 0

In [ ]:
def average_keystroke_savings_real(sample, word_counts, k=5):
    total = 0

    for word in sample:
        if len(word) < 3:
            continue
        total += keystroke_savings_real(word, word_counts, k)

    return total / len(sample)

In [ ]:
def top_k_accuracy_real(sample, word_counts, k=5):
    correct = 0
    total = 0

    for word in sample:
        found = False

        for i in range(1, len(word)):
            prefix = word[:i]
            suggestions = autocomplete(prefix, word_counts, k)

            if word in suggestions:
                correct += 1
                found = True
                break

        total += 1

    return correct / total

In [ ]:
accuracy = top_k_accuracy_real(sample, word_freq, k=5)
keystrokes = average_keystroke_savings_real(sample, word_freq, k=5)

print(f"Top-5 Accuracy: {accuracy:.4f}")
print(f"Average Keystroke Savings: {keystrokes:.2f}")

KeyboardInterrupt: 

In [ ]:
# simulate someone typing by iterating through a word
prefix = "c"
word = "dictionary"
for i in range(1, len(word) + 1):
    current = word[:i]
    print(f"Input: {current}")
    print("Suggestions:", autocomplete(current, word_freq))
    print()

Input: d
Suggestions: ['during', 'day', 'did', 'down', 'due']

Input: di
Suggestions: ['did', 'division', 'different', 'died', 'director']

Input: dic
Suggestions: ['dick', 'dictator', 'dictionary', 'dictated', 'dickens']

Input: dict
Suggestions: ['dictator', 'dictionary', 'dictated', 'dictatorship', 'dictatorial']

Input: dicti
Suggestions: ['dictionary', 'dictionaries', 'diction', 'dictionnaire']

Input: dictio
Suggestions: ['dictionary', 'dictionaries', 'diction', 'dictionnaire']

Input: diction
Suggestions: ['dictionary', 'dictionaries', 'diction', 'dictionnaire']

Input: dictiona
Suggestions: ['dictionary', 'dictionaries']

Input: dictionar
Suggestions: ['dictionary', 'dictionaries']

Input: dictionary
Suggestions: ['dictionary']



In [ ]:
# run the experiment 5 times

results = []

for _ in range(5):
    sample = random.sample(list(word_freq.keys()), 1000)

    acc = top_k_accuracy_real(sample, word_freq)
    ks = average_keystroke_savings_real(sample, word_freq)

    results.append((acc, ks))

avg_acc = sum(r[0] for r in results) / len(results)
avg_ks = sum(r[1] for r in results) / len(results)

print(f"Average Accuracy over runs: {avg_acc:.4f}")
print(f"Average Keystroke Savings: {avg_ks:.2f}")